---

# IF4061-Moview TMDb Data Preprocessing

*Pipeline v2 uses TMDB_movie_dataset_v11 with about 27k quality filtered films as primary source, joins tmdb_5000_credits for director and cast  where IDs overlap, and exports updated JSON files with month and cast fields.*

---

## Kelompok

- (Nama anggota)
- (Nama anggota)
- (Nama anggota)

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Parse & Clean**](#3)
4. [**GeoJSON Patch**](#4)
5. [**Export JSON**](#5)


---

# Introduction <a name="1"></a>

---

## Overview

v2 replaces the 4,803 film tmdb_5000 dataset with the larger TMDB v11 dataset with about 27k quality filtered films. Director and cast data come from the tmdb_5000_credits CSV joined by movie ID. People tab charts are scoped to the about 4,800 films where credits exist.

**Quality filter applied to v11** `status == Released`, `vote_count >= 50`, `year 1950 to 2024`.
**New fields added** `month` from 1 to 12, `release_season`, and `cast` with top 5 actor names.

## Outputs

All files written to `dataset/clean/`.

| File | Contents |
|------|----------|
| `manifest.json` | Available scopes, year bounds, and generated file paths |
| `geo_summary.json` | Country names, continents, total films, profit, and rating |
| `kpi.json` | Compact metric rows for hero and filters |
| `kpi_tiles.json` | Prepared range KPI rows for the active tab strip |
| `kpi_language.json` | Compact language rows for hero and filters |
| `movies.json` | Full movie detail fallback not used for startup |
| `scope/...` | Full aggregate shards by selected area |
| `countries.geojson` | Natural Earth polygons with patched ISO codes |

---

# Initialization <a name="2"></a>

---

In [1]:
#%pip install -q pandas numpy shapely tqdm

In [2]:
import ast
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings('ignore')

In [3]:
class Settings:
    BASE_DIR    = Path('d:/Akademik/ITB/Semester-6/IF4061-VisDat/Tubes/IF4061-Moview')
    RAW_DIR     = BASE_DIR / 'dataset' / 'raw'
    OUT_DIR     = BASE_DIR / 'dataset' / 'clean'

    V11_CSV     = RAW_DIR / 'TMDB_movie_dataset_v11.csv'
    CREDITS_CSV = RAW_DIR / 'tmdb_5000_credits.csv'
    GEOJSON     = RAW_DIR / 'ne_110m_admin_0_countries.geojson'

    # Quality filter
    MIN_VOTES   = 50
    YEAR_MIN    = 1950
    YEAR_MAX    = 2024

CFG = Settings()
CFG.OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output dir is', CFG.OUT_DIR)

Output dir is d:\Akademik\ITB\Semester-6\IF4061-VisDat\Tubes\IF4061-Moview\dataset\clean


---

# Parse & Clean <a name="3"></a>

---

## 3.1 Load v11 + quality filter

In [4]:
raw = pd.read_csv(CFG.V11_CSV, low_memory=False)
print(f'Raw v11 shape is {raw.shape}')

raw['year'] = pd.to_datetime(raw['release_date'], errors='coerce').dt.year
raw['month'] = pd.to_datetime(raw['release_date'], errors='coerce').dt.month

df = raw[
    (raw['status'] == 'Released') &
    (raw['vote_count'] >= CFG.MIN_VOTES) &
    (raw['year'] >= CFG.YEAR_MIN) &
    (raw['year'] <= CFG.YEAR_MAX)
].copy()

print(f'After quality filter there are {len(df)} films')

Raw v11 shape is (1432967, 24)
After quality filter there are 26925 films


## 3.2 Load credits, extract director + cast

In [5]:
credits_raw = pd.read_csv(CFG.CREDITS_CSV)
credits_raw = credits_raw.rename(columns={'movie_id': 'id'})

def parse_col(val):
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return []

def extract_director(val):
    for m in parse_col(val):
        if m.get('job') == 'Director':
            return m.get('name')
    return None

def extract_cast(val, n=5):
    members = parse_col(val)
    members.sort(key=lambda m: m.get('order', 999))
    return [m['name'] for m in members[:n] if 'name' in m]

credits_raw['director'] = credits_raw['crew'].apply(extract_director)
credits_raw['cast_list'] = credits_raw['cast'].apply(extract_cast)

credits_slim = credits_raw[['id', 'director', 'cast_list']]
overlap = set(df['id']).intersection(set(credits_slim['id']))
print(f'Credits overlap has {len(overlap)} films out of {len(df)} total')

Credits overlap has 4224 films out of 26925 total


## 3.3 Language code to name mapping

In [6]:
LANG_NAMES = {
    'af':'Afrikaans','ak':'Akan','am':'Amharic','ar':'Arabic','az':'Azerbaijani',
    'be':'Belarusian','bg':'Bulgarian','bm':'Bambara','bn':'Bengali','bo':'Tibetan',
    'bs':'Bosnian','ca':'Catalan','cs':'Czech','cy':'Welsh','da':'Danish',
    'de':'German','dz':'Dzongkha','el':'Greek','en':'English','eo':'Esperanto',
    'es':'Spanish','et':'Estonian','eu':'Basque','fa':'Persian','ff':'Fula',
    'fi':'Finnish','fr':'French','fy':'Frisian','ga':'Irish','gl':'Galician',
    'gu':'Gujarati','ha':'Hausa','he':'Hebrew','hi':'Hindi','hr':'Croatian',
    'hu':'Hungarian','hy':'Armenian','id':'Indonesian','ig':'Igbo','is':'Icelandic',
    'it':'Italian','iu':'Inuktitut','ja':'Japanese','jv':'Javanese','ka':'Georgian',
    'kk':'Kazakh','km':'Khmer','kn':'Kannada','ko':'Korean','ku':'Kurdish',
    'ky':'Kyrgyz','lb':'Luxembourgish','lo':'Lao','lt':'Lithuanian','lv':'Latvian',
    'mg':'Malagasy','mi':'Maori','mk':'Macedonian','ml':'Malayalam','mn':'Mongolian',
    'mr':'Marathi','ms':'Malay','mt':'Maltese','my':'Burmese','nb':'Norwegian',
    'ne':'Nepali','nl':'Dutch','nn':'Norwegian Nynorsk','no':'Norwegian','ny':'Chichewa',
    'om':'Oromo','or':'Odia','pa':'Punjabi','pl':'Polish','ps':'Pashto',
    'pt':'Portuguese','qu':'Quechua','ro':'Romanian','ru':'Russian','rw':'Kinyarwanda',
    'si':'Sinhala','sk':'Slovak','sl':'Slovenian','sm':'Samoan','sn':'Shona',
    'so':'Somali','sq':'Albanian','sr':'Serbian','ss':'Swati','st':'Sotho',
    'su':'Sundanese','sv':'Swedish','sw':'Swahili','ta':'Tamil','te':'Telugu',
    'tg':'Tajik','th':'Thai','ti':'Tigrinya','tk':'Turkmen','tl':'Filipino',
    'tn':'Tswana','to':'Tongan','tr':'Turkish','ts':'Tsonga','tt':'Tatar',
    'tw':'Twi','ug':'Uyghur','uk':'Ukrainian','ur':'Urdu','uz':'Uzbek',
    've':'Venda','vi':'Vietnamese','wo':'Wolof','xh':'Xhosa','yi':'Yiddish',
    'yo':'Yoruba','zh':'Chinese','zu':'Zulu','xx':'No Language','cn':'Cantonese',
}

def lang_name(code):
    if pd.isna(code): return None
    return LANG_NAMES.get(str(code).strip().lower(), str(code).upper())

print(f'Language map loaded with {len(LANG_NAMES)} entries')

Language map loaded with 120 entries


## 3.4 Parse columns

In [7]:
def split_csv(val):
    if pd.isna(val) or str(val).strip() == '':
        return []
    return [x.strip() for x in str(val).split(',') if x.strip()]

df['genres']               = df['genres'].apply(split_csv)
df['keywords']             = df['keywords'].apply(split_csv)
df['production_companies'] = df['production_companies'].apply(split_csv)
df['spoken_languages']     = df['spoken_languages'].apply(split_csv)
df['production_countries_raw'] = df['production_countries'].apply(split_csv)
df['language']             = df['original_language'].apply(lang_name)

df['primary_genre'] = df['genres'].apply(lambda g: g[0] if g else None)
print('Columns parsed.')

Columns parsed.


## 3.4 Map production country names to ISO_A2

In [8]:
with open(CFG.GEOJSON, encoding='utf-8') as f:
    _gj = json.load(f)

# Build name to ISO_A2 from GeoJSON names
name_to_iso = {}
iso_to_display_name = {}
iso_to_continent = {}
for feat in _gj['features']:
    p = feat['properties']
    iso = p.get('ISO_A2', '')
    if not iso or iso == '-99':
        iso = p.get('ISO_A2_EH', '')
    if iso and iso != '-99':
        iso_to_display_name[iso] = p.get('NAME_LONG') or p.get('NAME') or iso
        iso_to_continent[iso] = p.get('CONTINENT')
        for key in ('NAME', 'NAME_LONG', 'FORMAL_EN', 'BRK_NAME'):
            name = p.get(key, '')
            if name:
                name_to_iso[name] = iso

# Manual overrides for common v11 country name mismatches
name_to_iso.update({
    'United States of America': 'US',
    'United Kingdom':           'GB',
    'South Korea':              'KR',
    'Russia':                   'RU',
    'Czech Republic':           'CZ',
    'Iran':                     'IR',
    'Syria':                    'SY',
    'Bolivia':                  'BO',
    'Venezuela':                'VE',
    'Tanzania':                 'TZ',
    'Moldova':                  'MD',
    'Vietnam':                  'VN',
    'Hong Kong':                'HK',
    'Taiwan':                   'TW',
    'Palestine':                'PS',
    'Macedonia':                'MK',
    'Kosovo':                   'XK',
})
iso_to_display_name.update({
    'HK': 'Hong Kong',
    'RU': 'Russia',
    'TW': 'Taiwan',
    'XK': 'Kosovo',
})
iso_to_continent.update({
    'AW': 'North America', 'GP': 'North America',
    'HK': 'Asia',          'MT': 'Europe',
    'SG': 'Asia',          'TW': 'Asia',
})

def get_primary_iso(countries):
    for c in countries:
        iso = name_to_iso.get(c)
        if iso:
            return iso
    return None

df['primary_iso'] = df['production_countries_raw'].apply(get_primary_iso)
df['continent'] = df['primary_iso'].map(iso_to_continent)

matched = df['primary_iso'].notna().sum()
print(f'ISO matched {matched}/{len(df)} films ({matched/len(df)*100:.1f}%)')

ISO matched 26544/26925 films (98.6%)


## 3.5 Derived fields + credit join

In [9]:
df['budget']  = pd.to_numeric(df['budget'],  errors='coerce').replace(0, np.nan)
df['revenue'] = pd.to_numeric(df['revenue'], errors='coerce').replace(0, np.nan)
df['profit']  = df['revenue'] - df['budget']
df['decade']  = (df['year'] // 10 * 10).astype('Int64')

SEASON = {12:'Winter',1:'Winter',2:'Winter',
          3:'Spring',4:'Spring',5:'Spring',
          6:'Summer',7:'Summer',8:'Summer',
          9:'Fall',10:'Fall',11:'Fall'}
df['release_season'] = df['month'].map(SEASON)

# Join director + cast (null for non-overlapping rows)
df = df.merge(credits_slim, on='id', how='left')
df['cast_list'] = df['cast_list'].apply(lambda x: x if isinstance(x, list) else [])

print(f'Total films {len(df)}')
print(f'With director {df["director"].notna().sum()}')
print(f'With cast {df["cast_list"].apply(bool).sum()}')

Total films 26925
With director 4223
With cast 4224


## 3.6 Financial subset + profit/budget tiers

In [10]:
fin = df.dropna(subset=['budget', 'revenue']).copy()

q1 = float(fin['profit'].quantile(0.25))
q2 = float(fin['profit'].quantile(0.50))
q3 = float(fin['profit'].quantile(0.75))
print(f'Profit tiers Q1={q1/1e6:.1f}M  Q2={q2/1e6:.1f}M  Q3={q3/1e6:.1f}M')

def assign_profit_tier(profit):
    if pd.isna(profit): return None
    if profit < q1:     return 'Low'
    if profit < q2:     return 'Mid-Low'
    if profit < q3:     return 'Mid-High'
    return 'High'

def assign_budget_tier(budget):
    if pd.isna(budget):      return None
    if budget < 10_000_000:  return 'Low'
    if budget < 50_000_000:  return 'Mid'
    return 'High'

fin['profit_tier'] = fin['profit'].apply(assign_profit_tier)
fin['budget_tier'] = fin['budget'].apply(assign_budget_tier)
print(f'Financial subset has {len(fin)} films')

Profit tiers Q1=-2.5M  Q2=9.1M  Q3=52.4M
Financial subset has 7641 films


---

# GeoJSON Patch <a name="4"></a>

---

## 4.1 countries.geojson

Same patch as v1. It fixes `ISO_A2 = -99` using `ISO_A2_EH`, drops Antarctica, and drops open ocean.

In [11]:
with open(CFG.GEOJSON, encoding='utf-8') as f:
    geojson = json.load(f)

patched = []
kept = []
for feat in geojson['features']:
    p = feat['properties']
    if p.get('CONTINENT') in ('Antarctica', 'Seven seas (open ocean)'):
        continue
    if p.get('ISO_A2') == '-99' and p.get('ISO_A2_EH', '-99') != '-99':
        p['ISO_A2'] = p['ISO_A2_EH']
        patched.append(p['NAME'])
    kept.append(feat)

geojson['features'] = kept
out_path = CFG.OUT_DIR / 'countries.geojson'
out_path.write_text(json.dumps(geojson, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Kept {len(kept)} features and patched {len(patched)} values {patched}')

Kept 175 features and patched 3 values ['Norway', 'France', 'Kosovo']


## 4.2 continents.geojson

In [12]:
from shapely.geometry import shape, mapping
from shapely.ops import unary_union
from collections import defaultdict

with open(CFG.OUT_DIR / 'countries.geojson', encoding='utf-8') as f:
    _cj = json.load(f)

by_cont = defaultdict(list)
for feat in _cj['features']:
    cont = feat['properties'].get('CONTINENT')
    if cont:
        try:
            by_cont[cont].append(shape(feat['geometry']))
        except Exception:
            pass

cont_features = [
    {'type': 'Feature', 'properties': {'continent': cont}, 'geometry': mapping(unary_union(polys))}
    for cont, polys in by_cont.items()
]
cont_out_path = CFG.OUT_DIR / 'continents.geojson'
cont_out_path.write_text(
    json.dumps({'type': 'FeatureCollection', 'features': cont_features}, indent=2, ensure_ascii=False),
    encoding='utf-8',
)
print(f'continents.geojson has {len(cont_features)} continents')

continents.geojson has 6 continents


---

# Export JSON <a name="5"></a>

---

Same structure as v1 with additions `month`, `cast`, and `release_season`.

## 5.1 movies.json

In [13]:
def safe_float(v):
    if pd.isna(v):
        return None
    value = float(v)
    return round(value, 4) if np.isfinite(value) else None

def safe_int(v):
    return int(v) if pd.notna(v) else None

def sanitize_value(v):
    if isinstance(v, (np.floating, float)):
        value = float(v)
        return None if not np.isfinite(value) else value
    if isinstance(v, (np.integer, int)):
        return int(v)
    if isinstance(v, dict):
        return {k2: sanitize_value(v2) for k2, v2 in v.items()}
    if isinstance(v, list):
        return [sanitize_value(i) for i in v]
    return v

def sanitize_records(records):
    return [
        {k: sanitize_value(v) for k, v in row.items()}
        for row in records
    ]

def write_json(path, data):
    path.write_text(
        json.dumps(data, indent=2, ensure_ascii=False, allow_nan=False),
        encoding='utf-8',
    )

movie_cols = [
    'id', 'title', 'year', 'month', 'decade', 'genres', 'primary_genre',
    'budget', 'revenue', 'profit', 'vote_average', 'vote_count', 'popularity',
    'director', 'cast', 'production_companies', 'primary_iso', 'continent',
    'original_language', 'keywords',
]
movies_out = sanitize_records(
    df
    .assign(cast=df['cast_list'], original_language=df['language'])
    .loc[:, movie_cols]
    .to_dict('records')
)
write_json(CFG.OUT_DIR / 'movies.json', movies_out)
print(f'movies.json has {len(movies_out)} records')

movies.json has 26925 records


## 5.2 Aggregate helpers

In [14]:
METRIC_COLS = ['budget', 'revenue', 'profit', 'vote_average', 'vote_count', 'popularity']
BASE_COLS = ['scope_type', 'scope_id', 'year', 'id'] + METRIC_COLS

def add_scope_rows(source):
    base = source[source['year'].notna()].copy()
    frames = [base.assign(scope_type='world', scope_id='WORLD')]

    continent_rows = base[base['continent'].notna()].copy()
    if not continent_rows.empty:
        frames.append(continent_rows.assign(scope_type='continent', scope_id=continent_rows['continent']))

    country_rows = base[base['primary_iso'].notna()].copy()
    if not country_rows.empty:
        frames.append(country_rows.assign(scope_type='country', scope_id=country_rows['primary_iso']))

    return pd.concat(frames, ignore_index=True)

def metric_agg(frame, group_cols):
    if frame.empty:
        return []
    out = (
        frame
        .groupby(group_cols, dropna=False)
        .agg(
            film_count=('id', 'count'),
            budget_sum=('budget', 'sum'), budget_count=('budget', 'count'),
            revenue_sum=('revenue', 'sum'), revenue_count=('revenue', 'count'),
            profit_sum=('profit', 'sum'), profit_count=('profit', 'count'),
            rating_sum=('vote_average', 'sum'), rating_count=('vote_average', 'count'),
            vote_count_sum=('vote_count', 'sum'),
            popularity_sum=('popularity', 'sum'), popularity_count=('popularity', 'count'),
        )
        .reset_index()
    )
    return sanitize_records(out.to_dict('records'))

scoped = add_scope_rows(df)
financial_scoped = add_scope_rows(fin)
print(f'Scoped movie rows {len(scoped)}')

Scoped movie rows 80013


## 5.3 Core files and scoped shards

In [15]:
from datetime import datetime, timezone
import shutil

SCOPE_DIR = CFG.OUT_DIR / 'scope'
if SCOPE_DIR.exists():
    shutil.rmtree(SCOPE_DIR)
SCOPE_DIR.mkdir(parents=True, exist_ok=True)

def scope_slug(scope_id):
    return str(scope_id).replace(' ', '_').replace('/', '_')

def scope_dir(scope_type, scope_id):
    if scope_type == 'world':
        return SCOPE_DIR / 'world'
    if scope_type == 'continent':
        return SCOPE_DIR / 'continent' / scope_slug(scope_id)
    raise ValueError(f'Unsupported shard scope {scope_type}')

def add_shard_columns(frame):
    out = frame.copy()
    out['_shard_type'] = np.where(out['scope_type'].eq('world'), 'world', 'continent')
    out['_shard_id'] = np.where(
        out['scope_type'].eq('world'),
        'WORLD',
        np.where(
            out['scope_type'].eq('continent'),
            out['scope_id'],
            out['scope_id'].map(iso_to_continent),
        ),
    )
    return out[out['_shard_id'].notna()]

def write_scope_file(records, filename):
    frame = pd.DataFrame(records)
    if frame.empty:
        return []

    frame = add_shard_columns(frame)
    written = []
    for (shard_type, shard_id), group in tqdm(
        frame.groupby(['_shard_type', '_shard_id'], dropna=False),
        desc=f'Writing {filename}',
        leave=True,
    ):
        out_dir = scope_dir(shard_type, shard_id)
        out_dir.mkdir(parents=True, exist_ok=True)
        rel_path = out_dir.relative_to(CFG.OUT_DIR) / filename
        group = group.drop(columns=['_shard_type', '_shard_id'])
        write_json(CFG.OUT_DIR / rel_path, sanitize_records(group.to_dict('records')))
        written.append(str(rel_path).replace('\\', '/'))
    return written

def avg_from_sum_count(row, sum_col, count_col):
    count = row.get(count_col, 0)
    return row.get(sum_col, 0) / count if count else None

scope_rows = (
    scoped[['scope_type', 'scope_id']]
    .drop_duplicates()
    .sort_values(['scope_type', 'scope_id'])
)
shard_rows = scope_rows[scope_rows['scope_type'].isin(['world', 'continent'])].copy()

kpi_out = metric_agg(scoped, ['scope_type', 'scope_id', 'year'])
kpi_language_out = sanitize_records(
    scoped
    .groupby(['scope_type', 'scope_id', 'year', 'language'], dropna=False)
    .agg(film_count=('id', 'count'))
    .reset_index()
    .to_dict('records')
)

country_totals = pd.DataFrame(kpi_out)
country_totals = country_totals[country_totals['scope_type'] == 'country']
country_totals = (
    country_totals
    .groupby('scope_id', as_index=False)
    .agg(
        film_count=('film_count', 'sum'),
        profit_sum=('profit_sum', 'sum'), profit_count=('profit_count', 'sum'),
        rating_sum=('rating_sum', 'sum'), rating_count=('rating_count', 'sum'),
    )
)
geo_summary_out = []
for row in country_totals.to_dict('records'):
    iso = row['scope_id']
    geo_summary_out.append({
        'iso_a2': iso,
        'name': iso_to_display_name.get(iso, iso),
        'continent': iso_to_continent.get(iso),
        'film_count': row['film_count'],
        'avg_profit': avg_from_sum_count(row, 'profit_sum', 'profit_count'),
        'avg_rating': avg_from_sum_count(row, 'rating_sum', 'rating_count'),
    })

write_json(CFG.OUT_DIR / 'kpi.json', kpi_out)
write_json(CFG.OUT_DIR / 'kpi_language.json', kpi_language_out)
write_json(CFG.OUT_DIR / 'geo_summary.json', sanitize_records(geo_summary_out))

scope_files = {}
scope_files['kpi.json'] = write_scope_file(kpi_out, 'kpi.json')
scope_files['kpi_language.json'] = write_scope_file(kpi_language_out, 'kpi_language.json')

print(f'Core kpi rows {len(kpi_out)}')
print(f'Core language rows {len(kpi_language_out)}')
print(f'Geo summary rows {len(geo_summary_out)}')

Writing kpi_language.json: 100%|██████████| 7/7 [00:00<00:00, 21.34it/s]

Core kpi rows 2367
Core language rows 6387
Geo summary rows 115


## 5.4 Scope aggregates

In [16]:
genre_scoped = (
    scoped
    .explode('genres')
    .rename(columns={'genres': 'genre'})
)
genre_scoped = genre_scoped[genre_scoped['genre'].notna() & (genre_scoped['genre'] != '')]
genre_agg_out = metric_agg(genre_scoped, ['scope_type', 'scope_id', 'year', 'genre'])
scope_files['genre_agg.json'] = write_scope_file(genre_agg_out, 'genre_agg.json')

financial_grouped = financial_scoped.rename(columns={'primary_genre': 'genre'})
financial_agg_out = metric_agg(
    financial_grouped,
    ['scope_type', 'scope_id', 'year', 'genre', 'budget_tier', 'release_season', 'profit_tier'],
)
scope_files['financial_agg.json'] = write_scope_file(financial_agg_out, 'financial_agg.json')

director_entities = (
    scoped.loc[scoped['director'].notna(), BASE_COLS + ['director']]
    .rename(columns={'director': 'name'})
    .assign(entity_type='director')
)
studio_entities = (
    scoped.loc[:, BASE_COLS + ['production_companies']]
    .explode('production_companies')
    .rename(columns={'production_companies': 'name'})
    .assign(entity_type='studio')
)
studio_entities = studio_entities[studio_entities['name'].notna() & (studio_entities['name'] != '')]
cast_entities = (
    scoped.loc[:, BASE_COLS + ['cast_list']]
    .explode('cast_list')
    .rename(columns={'cast_list': 'name'})
    .assign(entity_type='cast')
)
cast_entities = cast_entities[cast_entities['name'].notna() & (cast_entities['name'] != '')]
people_df = pd.concat([director_entities, studio_entities, cast_entities], ignore_index=True)
people_agg_out = metric_agg(people_df, ['scope_type', 'scope_id', 'year', 'entity_type', 'name'])
scope_files['people_agg.json'] = write_scope_file(people_agg_out, 'people_agg.json')

print(f'Genre aggregate rows {len(genre_agg_out)}')
print(f'Financial aggregate rows {len(financial_agg_out)}')
print(f'People aggregate rows {len(people_agg_out)}')

Writing people_agg.json: 100%|██████████| 7/7 [00:16<00:00,  2.42s/it]

Genre aggregate rows 18811
Financial aggregate rows 18250
People aggregate rows 232890


## 5.5 KPI tile ranges

In [17]:
def safe_div(numerator, denominator):
    return numerator / denominator if denominator else None

def list_count(value):
    return len(value) if isinstance(value, list) else 0

def scope_key_filter(frame, scope_type, scope_id):
    return frame[(frame['scope_type'] == scope_type) & (frame['scope_id'] == scope_id)]

def metric_range_rows(metric_frame, tab, builder, domain_frame=None):
    rows = []
    source = metric_frame if domain_frame is None else domain_frame
    if source.empty:
        return rows
    numeric_cols = [c for c in metric_frame.columns if c not in ['scope_type', 'scope_id', 'year']]
    metric_groups = {
        key: group.copy()
        for key, group in metric_frame.groupby(['scope_type', 'scope_id'], dropna=False)
    }
    groups = list(source.groupby(['scope_type', 'scope_id'], dropna=False))
    for (scope_type, scope_id), domain_group in tqdm(groups, desc=f'Building {tab} KPI tile ranges', leave=True):
        years = sorted(domain_group['year'].astype(int).unique().tolist())
        metric_group = metric_groups.get((scope_type, scope_id))
        if metric_group is None:
            aligned = pd.DataFrame(0, index=years, columns=numeric_cols)
        else:
            metric_group['year'] = metric_group['year'].astype(int)
            aligned = (
                metric_group.groupby('year', as_index=True)[numeric_cols]
                .sum()
                .reindex(years, fill_value=0)
            )
        cumulative = {col: [0] + aligned[col].fillna(0).cumsum().tolist() for col in numeric_cols}
        for start_index, from_year in enumerate(years):
            for end_index in range(start_index, len(years)):
                to_year = years[end_index]
                values = {
                    col: cumulative[col][end_index + 1] - cumulative[col][start_index]
                    for col in numeric_cols
                }
                rows.append(builder(scope_type, scope_id, from_year, to_year, values))
    return rows

kpi_frame = pd.DataFrame(kpi_out)
financial_year_frame = pd.DataFrame(metric_agg(financial_scoped, ['scope_type', 'scope_id', 'year']))
language_frame = pd.DataFrame(kpi_language_out)
genre_frame = pd.DataFrame(genre_agg_out)
people_frame = pd.DataFrame(people_agg_out)

popularity_rows = metric_range_rows(
    kpi_frame,
    'popularity',
    lambda scope_type, scope_id, from_year, to_year, v: {
        'scope_type': scope_type,
        'scope_id': scope_id,
        'from_year': from_year,
        'to_year': to_year,
        'tab': 'popularity',
        'films': v['film_count'],
        'avg_popularity': safe_div(v['popularity_sum'], v['popularity_count']),
        'avg_rating': safe_div(v['rating_sum'], v['rating_count']),
        'vote_count': v['vote_count_sum'],
    },
)

financial_rows = metric_range_rows(
    financial_year_frame,
    'financial',
    lambda scope_type, scope_id, from_year, to_year, v: {
        'scope_type': scope_type,
        'scope_id': scope_id,
        'from_year': from_year,
        'to_year': to_year,
        'tab': 'financial',
        'films': v['film_count'] or None,
        'avg_budget': safe_div(v['budget_sum'], v['budget_count']),
        'avg_profit': safe_div(v['profit_sum'], v['profit_count']),
        'avg_roi': safe_div(v['revenue_sum'], v['budget_sum']),
    },
    domain_frame=kpi_frame,
)

def category_range_maps(frame, years, value_col):
    if frame.empty:
        return {}, {}
    work = frame[['year', value_col, 'film_count']].dropna(subset=[value_col]).copy()
    if work.empty:
        return {}, {}
    pivot = (
        work
        .pivot_table(index='year', columns=value_col, values='film_count', aggfunc='sum', fill_value=0)
        .reindex(years, fill_value=0)
    )
    labels = pivot.columns.to_numpy(dtype=object)
    cumulative = np.vstack([
        np.zeros((1, len(labels)), dtype=np.int64),
        np.cumsum(pivot.to_numpy(dtype=np.int64), axis=0),
    ])
    top_map = {}
    count_map = {}
    for start_index, from_year in enumerate(years):
        totals = cumulative[start_index + 1:] - cumulative[start_index]
        active_counts = np.count_nonzero(totals, axis=1)
        top_indices = np.argmax(totals, axis=1) if len(labels) else []
        top_values = np.max(totals, axis=1) if len(labels) else []
        for offset, active_count in enumerate(active_counts):
            to_year = years[start_index + offset]
            key = (from_year, to_year)
            count_map[key] = None if active_count == 0 else int(active_count)
            top_map[key] = None if top_values[offset] <= 0 else labels[top_indices[offset]]
    return count_map, top_map

def entity_range_counts(frame, years, entity_type):
    work = frame[frame['entity_type'] == entity_type][['year', 'name']].drop_duplicates().copy()
    if work.empty:
        return {}
    work = work[work['year'].isin(years)]
    if work.empty:
        return {}
    year_pos = {year: i for i, year in enumerate(years)}
    year_codes = work['year'].map(year_pos).to_numpy(dtype=np.int64)
    name_codes = pd.factorize(work['name'], sort=False)[0]
    matrix = np.zeros((len(years), int(name_codes.max()) + 1), dtype=np.uint8)
    matrix[year_codes, name_codes] = 1
    cumulative = np.vstack([
        np.zeros((1, matrix.shape[1]), dtype=np.uint16),
        np.cumsum(matrix, axis=0, dtype=np.uint16),
    ])
    count_map = {}
    for start_index, from_year in enumerate(years):
        active = np.count_nonzero(cumulative[start_index + 1:] - cumulative[start_index], axis=1)
        for offset, count in enumerate(active):
            count_map[(from_year, years[start_index + offset])] = None if count == 0 else int(count)
    return count_map

genre_rows = []
scope_groups = list(kpi_frame.groupby(['scope_type', 'scope_id'], dropna=False))
for (scope_type, scope_id), base_group in tqdm(scope_groups, desc='Building genre KPI tile ranges', leave=True):
    base_group = base_group.sort_values('year')
    years = base_group['year'].astype(int).tolist()
    film_cumulative = [0] + base_group['film_count'].fillna(0).cumsum().tolist()
    genre_count_map, top_genre_map = category_range_maps(scope_key_filter(genre_frame, scope_type, scope_id), years, 'genre')
    _, top_language_map = category_range_maps(scope_key_filter(language_frame, scope_type, scope_id), years, 'language')
    for start_index, from_year in enumerate(years):
        for end_index in range(start_index, len(years)):
            to_year = years[end_index]
            key = (from_year, to_year)
            genre_rows.append({
                'scope_type': scope_type,
                'scope_id': scope_id,
                'from_year': from_year,
                'to_year': to_year,
                'tab': 'genre',
                'films': film_cumulative[end_index + 1] - film_cumulative[start_index],
                'active_genres': genre_count_map.get(key),
                'top_genre': top_genre_map.get(key),
                'top_language': top_language_map.get(key),
            })

people_names = people_df[['scope_type', 'scope_id', 'year', 'entity_type', 'name']].drop_duplicates()

people_rows = []
for (scope_type, scope_id), base_group in tqdm(scope_groups, desc='Building people KPI tile ranges', leave=True):
    base_group = base_group.sort_values('year')
    years = base_group['year'].astype(int).tolist()
    film_cumulative = [0] + base_group['film_count'].fillna(0).cumsum().tolist()
    scope_people = scope_key_filter(people_names, scope_type, scope_id)
    studio_count_map = entity_range_counts(scope_people, years, 'studio')
    director_count_map = entity_range_counts(scope_people, years, 'director')
    cast_count_map = entity_range_counts(scope_people, years, 'cast')
    for start_index, from_year in enumerate(years):
        for end_index in range(start_index, len(years)):
            to_year = years[end_index]
            key = (from_year, to_year)
            people_rows.append({
                'scope_type': scope_type,
                'scope_id': scope_id,
                'from_year': from_year,
                'to_year': to_year,
                'tab': 'people',
                'films': film_cumulative[end_index + 1] - film_cumulative[start_index],
                'studio_count': studio_count_map.get(key),
                'director_count': director_count_map.get(key),
                'cast_count': cast_count_map.get(key),
            })

kpi_tiles_out = sanitize_records(popularity_rows + financial_rows + genre_rows + people_rows)
kpi_tiles_core = [row for row in kpi_tiles_out if row['scope_type'] == 'world']
write_json(CFG.OUT_DIR / 'kpi_tiles.json', kpi_tiles_core)
scope_files['kpi_tiles.json'] = write_scope_file(kpi_tiles_out, 'kpi_tiles.json')

print(f'KPI tile range rows {len(kpi_tiles_out)}')

Writing kpi_tiles.json: 100%|██████████| 7/7 [00:17<00:00,  2.51s/it]

KPI tile range rows 226744


## 5.6 Scope pair keyword and movie files

In [18]:
pair_source_cols = BASE_COLS + ['genres']
pair_source = scoped.loc[
    scoped['genres'].apply(lambda g: isinstance(g, list) and len(g) >= 2),
    pair_source_cols,
]

pair_rows = []
for row in tqdm(pair_source.itertuples(index=False), total=len(pair_source), desc='Building genre pair rows', leave=True):
    genres = sorted(set(row.genres))
    for i, genre_a in enumerate(genres):
        for genre_b in genres[i + 1:]:
            pair_rows.append({
                'scope_type': row.scope_type, 'scope_id': row.scope_id, 'year': row.year,
                'genre_a': genre_a, 'genre_b': genre_b, 'id': row.id,
                'budget': row.budget, 'revenue': row.revenue, 'profit': row.profit,
                'vote_average': row.vote_average, 'vote_count': row.vote_count,
                'popularity': row.popularity,
            })
pair_df = pd.DataFrame(pair_rows)
genre_pair_agg_out = metric_agg(pair_df, ['scope_type', 'scope_id', 'year', 'genre_a', 'genre_b'])
scope_files['genre_pair_agg.json'] = write_scope_file(genre_pair_agg_out, 'genre_pair_agg.json')

keyword_df = (
    scoped.loc[:, ['scope_type', 'scope_id', 'year', 'genres', 'keywords']]
    .explode('genres')
    .explode('keywords')
    .rename(columns={'genres': 'genre', 'keywords': 'keyword'})
)
keyword_df = keyword_df[
    keyword_df['genre'].notna() & (keyword_df['genre'] != '') &
    keyword_df['keyword'].notna() & (keyword_df['keyword'] != '')
]
keyword_agg = (
    keyword_df
    .groupby(['scope_type', 'scope_id', 'year', 'genre', 'keyword'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values(['scope_type', 'scope_id', 'year', 'genre', 'count'], ascending=[True, True, True, True, False])
)
keyword_agg_out = sanitize_records(keyword_agg.to_dict('records'))
scope_files['keyword_agg.json'] = write_scope_file(keyword_agg_out, 'keyword_agg.json')

top_movie_cols = [
    'scope_type', 'scope_id', 'id', 'title', 'year', 'month', 'primary_genre',
    'primary_iso', 'continent', 'budget', 'revenue', 'profit', 'vote_average',
    'vote_count', 'popularity',
]
top_movies_out = sanitize_records(scoped.loc[:, top_movie_cols].to_dict('records'))
scope_files['top_movies.json'] = write_scope_file(top_movies_out, 'top_movies.json')

print(f'Genre pair aggregate rows {len(genre_pair_agg_out)}')
print(f'Keyword aggregate rows {len(keyword_agg_out)}')
print(f'Top movie scoped rows {len(top_movies_out)}')

Writing top_movies.json: 100%|██████████| 7/7 [00:04<00:00,  1.55it/s]

Genre pair aggregate rows 46534
Keyword aggregate rows 1085628
Top movie scoped rows 80013


## 5.7 Manifest

In [19]:
manifest_scopes = []
for row in shard_rows.to_dict('records'):
    scope_type = row['scope_type']
    scope_id = row['scope_id']
    rel_dir = scope_dir(scope_type, scope_id).relative_to(CFG.OUT_DIR)
    manifest_scopes.append({
        'scope_type': scope_type,
        'scope_id': scope_id,
        'path': str(rel_dir).replace('\\', '/'),
    })

manifest = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'year_min': int(df['year'].min()),
    'year_max': int(df['year'].max()),
    'scopes': manifest_scopes,
    'scope_files': sorted(scope_files.keys()),
    'core_files': ['manifest.json', 'geo_summary.json', 'kpi.json', 'kpi_tiles.json', 'kpi_language.json', 'movies.json', 'countries.geojson', 'continents.geojson'],
}

write_json(CFG.OUT_DIR / 'manifest.json', manifest)

print(f'Manifest scopes {len(manifest_scopes)}')

Manifest scopes 7


#### Insights

> All output files written to `dataset/clean/`. People data scoped to about 4,800 credit joined rows and all other charts use the full about 27k filtered set.